# 02 — Data Cleaning
**Bluestock Mutual Fund Analytics Capstone**

Covers:
1. Load raw datasets
2. Inspect data types, nulls, duplicates
3. Clean NAV history — parse dates, compute daily returns
4. Clean scheme performance — derived metrics, risk encoding
5. Clean investor transactions — date parts, amount buckets, KYC flag
6. Data quality validation
7. Save cleaned files to `data/processed/`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
BASE = Path('..') 
RAW  = BASE / 'data' / 'raw'
PROC = BASE / 'data' / 'processed'
PROC.mkdir(parents=True, exist_ok=True)
print('Paths configured ✅')

Paths configured ✅


---
## 1. Load Raw Datasets

In [2]:
fund_master  = pd.read_csv(RAW / '01_fund_master.csv', parse_dates=['launch_date'])
nav_raw      = pd.read_csv(RAW / '02_nav_history.csv', parse_dates=['date'])
performance  = pd.read_csv(RAW / '07_scheme_performance.csv')
transactions = pd.read_csv(RAW / '08_investor_transactions.csv', parse_dates=['transaction_date'])

datasets = {'fund_master': fund_master, 'nav_raw': nav_raw,
            'performance': performance, 'transactions': transactions}
for name, df in datasets.items():
    print(f'{name:<15} {df.shape}  nulls={df.isnull().sum().sum()}  dups={df.duplicated().sum()}')

fund_master     (40, 15)  nulls=0  dups=0
nav_raw         (46000, 3)  nulls=0  dups=0
performance     (40, 19)  nulls=0  dups=0
transactions    (32778, 13)  nulls=0  dups=0


---
## 2. Null & Duplicate Inspection

In [3]:
for name, df in datasets.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls):
        print(f'\n{name} — missing values:')
        print(nulls)
    else:
        print(f'{name}: no nulls ✅')

fund_master: no nulls ✅
nav_raw: no nulls ✅
performance: no nulls ✅
transactions: no nulls ✅


---
## 3. Clean NAV History

In [4]:
nav = nav_raw.copy()
nav['date'] = pd.to_datetime(nav['date'])
nav.dropna(subset=['nav'], inplace=True)
nav.sort_values(['amfi_code', 'date'], inplace=True)
nav.reset_index(drop=True, inplace=True)

# Daily return
nav['daily_return_pct'] = nav.groupby('amfi_code')['nav'].pct_change().mul(100).round(4)

# 30-day rolling volatility
nav['rolling_vol_30d'] = (
    nav.groupby('amfi_code')['daily_return_pct']
    .transform(lambda x: x.rolling(30, min_periods=10).std() * np.sqrt(252))
    .round(4)
)

# Merge scheme metadata
master_cols = ['amfi_code','scheme_name','fund_house','category','sub_category','plan']
nav = nav.merge(fund_master[master_cols], on='amfi_code', how='left')

print(f'NAV clean shape: {nav.shape}')
print(nav.dtypes)
print(f'\nNull daily_return (first row per scheme): {nav["daily_return_pct"].isna().sum()}')
nav.head(3)

NAV clean shape: (46000, 10)
amfi_code                    int64
date                datetime64[ns]
nav                        float64
daily_return_pct           float64
rolling_vol_30d            float64
scheme_name                 object
fund_house                  object
category                    object
sub_category                object
plan                        object
dtype: object

Null daily_return (first row per scheme): 40


,amfi_code,date,nav,daily_return_pct,rolling_vol_30d,scheme_name,fund_house,category,sub_category,plan
0,100016,2022-01-03,520.4608,NaN,NaN,HDFC Top 100 Fund - Regular Plan - Growth,HDFC Mutual Fund,Equity,Large Cap,Regular
1,100016,2022-01-04,515.0971,-1.0306,NaN,HDFC Top 100 Fund - Regular Plan - Growth,HDFC Mutual Fund,Equity,Large Cap,Regular
2,100016,2022-01-05,521.7239,1.2865,NaN,HDFC Top 100 Fund - Regular Plan - Growth,HDFC Mutual Fund,Equity,Large Cap,Regular


---
## 4. Clean Scheme Performance

In [5]:
perf = performance.copy()

# Merge with fund_master
master_extra = fund_master[['amfi_code','launch_date','benchmark','risk_category',
                             'min_sip_amount','sebi_category_code']]
perf = perf.merge(master_extra, on='amfi_code', how='left')

# Risk encoding
risk_map = {'Low':1,'Moderately Low':2,'Moderate':3,
            'Moderately High':4,'High':5,'Very High':6}
perf['risk_score'] = perf['risk_grade'].map(risk_map)

# Derived metrics
perf['excess_return']     = (perf['return_3yr_pct'] - perf['benchmark_3yr_pct']).round(4)
perf['information_ratio'] = (perf['alpha'] / perf['std_dev_ann_pct']).round(4)

# Null check
nulls = perf.isnull().sum()
print('Null counts (performance_clean):')
print(nulls[nulls > 0] if nulls.any() else 'None ✅')
print(f'\nShape: {perf.shape}')
perf[['amfi_code','scheme_name','return_3yr_pct','sharpe_ratio',
       'alpha','risk_score','excess_return']].head(5)

Null counts (performance_clean):
None ✅

Shape: (40, 27)


,amfi_code,scheme_name,return_3yr_pct,sharpe_ratio,alpha,risk_score,excess_return
0,119551,SBI Bluechip Fund - Regular Plan - Growth,12.36,0.88,0.87,3,0.87
1,119552,SBI Bluechip Fund - Direct Plan - Growth,11.30,0.81,1.78,3,1.78
2,119598,SBI Small Cap Fund - Regular Plan - Growth,23.39,0.94,1.23,6,1.23
3,119599,SBI Small Cap Fund - Direct Plan - Growth,23.14,0.93,1.13,6,1.13
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,6.07,1.52,1.60,1,1.60


---
## 5. Clean Investor Transactions

In [6]:
txn = transactions.copy()
txn['amount_inr']  = pd.to_numeric(txn['amount_inr'], errors='coerce')
txn['transaction_date'] = pd.to_datetime(txn['transaction_date'])
txn.dropna(subset=['amount_inr','transaction_date'], inplace=True)

txn['year']        = txn['transaction_date'].dt.year
txn['month']       = txn['transaction_date'].dt.month
txn['quarter']     = txn['transaction_date'].dt.quarter
txn['month_label'] = txn['transaction_date'].dt.to_period('M').astype(str)
txn['kyc_verified']= txn['kyc_status'] == 'Verified'
txn['amount_bucket'] = pd.cut(
    txn['amount_inr'],
    bins=[0, 5000, 50000, 200000, float('inf')],
    labels=['Micro (<5K)','Small (5K-50K)','Mid (50K-2L)','Large (>2L)']
)
txn = txn.merge(fund_master[['amfi_code','scheme_name','fund_house',
                              'category','sub_category']], on='amfi_code', how='left')

print(f'Transactions clean shape: {txn.shape}')
print(f'Unique investors: {txn["investor_id"].nunique():,}')
txn.head(3)

Transactions clean shape: (32778, 23)
Unique investors: 5,000


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,...,year,month,quarter,month_label,kyc_verified,amount_bucket,scheme_name,fund_house,category,sub_category
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,...,2024,1,1,2024-01,True,Micro (<5K),Axis Bluechip Fund - Regular - Growth,Axis Mutual Fund,Equity,Large Cap
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,...,2024,1,1,2024-01,True,Large (>2L),Mirae Asset Large Cap Fund - Regular - Growth,Mirae Asset MF,Equity,Large Cap
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,...,2024,1,1,2024-01,True,Micro (<5K),Nippon India Gilt Securities Fund - Regular - ...,Nippon India MF,Debt,Gilt


---
## 6. Data Quality Validation

In [7]:
checks = {
    'NAV no nulls in nav col'         : nav['nav'].isnull().sum() == 0,
    'NAV sorted by date'              : nav.groupby('amfi_code')['date'].is_monotonic_increasing.all(),
    'Perf risk_score fully mapped'    : perf['risk_score'].isnull().sum() == 0,
    'Perf excess_return computed'     : 'excess_return' in perf.columns,
    'Txn amount all positive'         : (txn['amount_inr'] > 0).all(),
    'Txn date parsed correctly'       : pd.api.types.is_datetime64_any_dtype(txn['transaction_date']),
    'Txn amount_bucket no nulls'      : txn['amount_bucket'].isnull().sum() == 0,
}

print('DATA QUALITY VALIDATION')
print('=' * 50)
for check, result in checks.items():
    icon = '✅' if result else '❌'
    print(f'  {icon}  {check}')
print('\nAll checks:', 'PASSED ✅' if all(checks.values()) else 'SOME FAILED ❌')

DATA QUALITY VALIDATION
  ✅  NAV no nulls in nav col
  ✅  NAV sorted by date
  ✅  Perf risk_score fully mapped
  ✅  Perf excess_return computed
  ✅  Txn amount all positive
  ✅  Txn date parsed correctly
  ✅  Txn amount_bucket no nulls

All checks: PASSED ✅


---
## 7. Save Cleaned Files

In [8]:
nav.to_csv(PROC / 'nav_history_clean.csv',    index=False)
perf.to_csv(PROC / 'performance_clean.csv',   index=False)
txn.to_csv(PROC / 'transactions_clean.csv',   index=False)

print('Saved:')
for fname in ['nav_history_clean.csv','performance_clean.csv','transactions_clean.csv']:
    p = PROC / fname
    print(f'  {fname:<35} {p.stat().st_size//1024} KB')

Saved:
  nav_history_clean.csv               5661 KB
  performance_clean.csv               8 KB
  transactions_clean.csv              6642 KB
